In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so 'src' is importable
repo_root = str(Path.cwd().parents[1]) if Path.cwd().name == 'research_questions' else str(Path.cwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
from src.analysis.get_results import get_pandas_dataset
from src.analysis.dataset_graphs import plot_graphs_of_dataset_loc
from src.analysis.get_tables_results import create_table_cleaned
from src.analysis.get_results import bar_chart_cleaned,bar_chart_fix_position_cleaned
from src.analysis.get_results import sucess_vs_position_cleaned, get_latex_table_with_verif_stats
from src.analysis.get_results import compute_stats_tests

import src.config as gl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
RESULT_DIR = gl.BASE_PATH / "results/dafny_llm_results_rq2__loc_strategy"
DATASET_DIR = gl.DAFNY_ASSERTION_DATASET

print(DATASET_DIR)
print(RESULT_DIR)
verif_data_pd = get_pandas_dataset(DATASET_DIR, RESULT_DIR)
verif_data_pd  = verif_data_pd.assign(success=lambda d: d['verif_sucess'] > 0) 

In [ ]:
# Graphs of position evaluation
test_models = {
    "claude-haiku-4.5__loc_LLM_inf_LLM": "LLM$_{fl}$",
    "claude-haiku-4.5__loc_LLM_EXAMPLE_inf_LLM_sExPos_DYNAMIC_nExPos_3_aExPos_0.25": "LLMEX$_{fl}$",
    "claude-haiku-4.5__loc_LAUREL_inf_LLM": "Laurel$_{fl}$",
    "claude-haiku-4.5__loc_LAUREL_BETTER_inf_LLM": "Laurel$_{fl+}$",
    "claude-haiku-4.5__loc_HYBRID_inf_LLM_sExPos_DYNAMIC_nExPos_3_aExPos_0.25": "Hybrid$_{fl}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM": "GrTru$_{fl}$"
}

w1d = verif_data_pd[verif_data_pd["benchmark"] == "w/o-1"]
w2d =  verif_data_pd[verif_data_pd["benchmark"] == "w/o-2"]
#verif_data_pd = verif_data_pd[verif_data_pd["benchmark"] != "w/o-1"]

images_p = gl.BASE_PATH / "images"

info = get_latex_table_with_verif_stats(verif_data_pd, "Verification success rate for each approach for each category of benchmarks for the position retrieval strategy without examples (\enoex).", "tbl:assertion-inference-verification-position", test_models)
print(info)

bar_chart_fix_position_cleaned(w1d ,"DOUBLE",   test_models, images_p / "rq2__bar_chart_pos_w1d.pdf")
sucess_vs_position_cleaned(w1d ,"DOUBLE",   test_models, images_p / "rq2__sucess_vs_bar_chart_w1d.pdf")

bar_chart_fix_position_cleaned(w2d ,"DOUBLE",   test_models, images_p / "rq2__bar_chart_pos_w2d.pdf")
sucess_vs_position_cleaned(w2d ,"DOUBLE",   test_models, images_p / "rq2__sucess_vs_bar_chart_w2d.pdf")

bar_chart_fix_position_cleaned(verif_data_pd ,"DOUBLE",   test_models, images_p / "rq2__bar_chart_pos_combined.pdf")
sucess_vs_position_cleaned(verif_data_pd ,"DOUBLE",   test_models, images_p / "rq2__sucess_vs_bar_chart_combined.pdf")


In [ ]:
verif_renamed = verif_data_pd.copy()
verif_renamed["llm"] = verif_renamed["llm"].map(test_models)

print("##### Comparing LLM$_{fl}$ with LLMEX$_{fl}$ #####")
compute_stats_tests("LLM$_{fl}$","LLMEX$_{fl}$", verif_renamed)

print("##### Comparing Laurel$_{fl}$ with Laurel$_{fl+}$ #####")
compute_stats_tests("Laurel$_{fl}$","Laurel$_{fl+}$", verif_renamed)

print("##### Comparing LLMEX$_{fl}$ with Laurel$_{fl+}$ #####")
compute_stats_tests("LLMEX$_{fl}$","Laurel$_{fl+}$", verif_renamed)

print("##### Comparing LLMEX$_{fl}$ with Hybrid$_{fl}$ #####")
compute_stats_tests("LLMEX$_{fl}$","Hybrid$_{fl}$", verif_renamed)

print("##### Comparing Hybrid$_{fl}$ with Laurel$_{fl+}$ #####")
compute_stats_tests("Hybrid$_{fl}$","Laurel$_{fl+}$", verif_renamed)